In [ ]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# The dataset Q1_data.csv has been successfully loaded into a pandas DataFrame named df from the /content/ directory. This DataFrame is now ready for further inspection and analysis.

import pandas as pd

# Read the CSV file into a pandas DataFrame named df
df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

print("Dataset loaded successfully into DataFrame 'df' from the /content/ directory.")

In [ ]:
# Task 2: Write your code here:

# The df.head() method displays the first few rows of the DataFrame, which is useful for quickly verifying that the data has been loaded correctly and to get a preliminary look at its structure and content.

display(df.head())


In [ ]:
# Task 3: Write your code here:

#The `df.info()` method provides a concise summary of the DataFrame, including the data types of each column, the number of non-null values, and memory usage. This helps in understanding the data structure and identifying potential issues like missing values or incorrect data types.

display(df.info())

In [ ]:
# Task 4: Write your code here:

# The `df.describe()` method generates descriptive statistics that summarize the central tendency, dispersion, and shape of a dataset's distribution, excluding NaN values. This is useful for quickly understanding the main characteristics of numerical data.

display(df.describe())

In [ ]:
# Task 5: Write your code here:

# This code generates a histogram of the 'Delivery_Time' column, showing its distribution. The Kernel Density Estimate (KDE) overlay helps visualize the shape of the distribution, while the title and labels make the plot understandable.

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'], kde=True, bins=30)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Task 1: Write your code here:

# The 'Order_ID' column is unique for each entry and typically serves no analytical purpose in prediction models.
# Dropping it helps reduce dimensionality and ensures it doesn't accidentally get treated as a feature.

df = df.drop('Order_ID', axis=1)

print("Dropped 'Order_ID' column.")
display(df.head())

In [ ]:
# Task 2: Write your code here:

# Let's first see how many missing values we have in each column.
print("Missing values before handling:")
display(df.isnull().sum())

# For text-based categories like Weather, Traffic_Level, and Time_of_Day, it's often best to fill missing spots with the most common value (the 'mode').
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    if df[col].isnull().any():
        mode_value = df[col].mode()[0]
        df[col].fillna(mode_value, inplace=True)
        print(f"Filled missing values in '{col}' with its most frequent option: {mode_value}")

# For 'Courier_Experience_yrs', which is a number, using the middle value (the 'median') is a good way to fill in gaps without being too affected by extreme values.
if df['Courier_Experience_yrs'].isnull().any():
    median_value = df['Courier_Experience_yrs'].median()
    df['Courier_Experience_yrs'].fillna(median_value, inplace=True)
    print(f"Filled missing values in 'Courier_Experience_yrs' with the middle experience value: {median_value}")

# Our main goal is to predict 'Delivery_Time'. If this value is missing, that row isn't useful for training, so we'll remove those rows.
initial_rows = len(df)
df.dropna(subset=['Delivery_Time'], inplace=True)
dropped_rows = initial_rows - len(df)
print(f"Removed {dropped_rows} rows because 'Delivery_Time' was missing.")

# Now, let's check again to make sure all missing values are gone.
print("\nMissing values after handling:")
display(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:

# Let's find out if there are any exact duplicate rows. Duplicates can sometimes mess up our model training.
duplicates_before = df.duplicated().sum()
print(f"Number of duplicate rows before removal: {duplicates_before}")

if duplicates_before > 0:
    df.drop_duplicates(inplace=True)
    print("Okay, we found and removed the duplicate rows.")
    duplicates_after = df.duplicated().sum()
    print(f"Number of duplicate rows after removal: {duplicates_after}")
else:
    print("Good news! No duplicate rows were found in our dataset.")

In [ ]:
# Task 4: Write your code here:

# Our model needs numbers, so we need to convert text categories like 'Weather' or 'Traffic_Level' into numbers.
# One-Hot Encoding creates new columns for each unique category, turning them into 0s and 1s.
# We use 'drop_first=True' to avoid a common issue called multicollinearity.
categorical_cols = df.select_dtypes(include='object').columns
print(f"Columns that look like categories to encode: {list(categorical_cols)}")

if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print("Categorical columns have been transformed into numbers using One-Hot Encoding.")
else:
    print("No text-based (categorical) columns found that needed encoding.")

display(df.head())

In [ ]:
# Task 5: Write your code here:

# Imagine comparing apples and oranges – if one feature has huge numbers and another has tiny ones, our model might get confused.
# Scaling makes sure all our numerical features are on a similar playing field (mean of 0, standard deviation of 1).
from sklearn.preprocessing import StandardScaler

# First, we separate our prediction target (Delivery_Time) from the features we'll use to predict it.
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

# Now, we get our StandardScaler ready.
scaler = StandardScaler()

# We 'fit' the scaler to our features and then 'transform' them.
X_scaled = scaler.fit_transform(X)

# To keep things tidy and understandable, we put our scaled features back into a DataFrame.
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("All numerical features have been scaled so they're all on a similar scale.")
display(X_scaled.head())

In [ ]:
# Task 6: Write your code here:

# For prediction tasks like ours (where we predict a number, 'Delivery_Time'), 'imbalance' isn't really about having too many of one category like in classification.
# It's more about whether the numbers we're trying to predict are very skewed or have unusual outliers.
# We already looked at the distribution of 'Delivery_Time' in Part 1 with a histogram. That's usually how we check this for regression.

print("For this kind of prediction task (regression), 'target imbalance' isn't about having uneven groups like in classification. We already checked the overall shape of 'Delivery_Time' in Part 1. If it were extremely skewed, we might consider transforming it, but for now, the distribution was visualized.")

# If we wanted to get super technical, we could calculate skewness or look at a box plot, but the histogram gives us a good sense.
# print(f"Target skewness: {y.skew():.2f}")
# plt.figure(figsize=(8, 5))
# sns.boxplot(y=y)
# plt.title('Box Plot of Delivery Time')
# plt.show()

In [ ]:
# Task 1: Write your code here:

# We already prepared X as X_scaled (scaled features) and y (Delivery_Time) in the previous data cleaning step.
# So, X_scaled contains our features ready for modeling, and y is our target.
# X will refer to X_scaled for consistency.

X = X_scaled

In [ ]:
# Task 2,3,4,5: Write your code here:

# For regression tasks, KFold is the appropriate cross-validation strategy as we are not dealing with discrete classes.
from sklearn.model_selection import KFold

# We'll use 5 folds for cross-validation to get a robust evaluation of our model.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Let's keep track of our MAE scores from each fold.
mae_scores = []

# Loop through each fold generated by KFold
for train_index, test_index in kf.split(X):
    # Split the data for this fold into training and testing sets
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize our RandomForestRegressor model
    # We'll use a random_state for reproducibility
    model = RandomForestRegressor(random_state=42)

    # Train the model on the training data
    model.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = model.predict(X_test)

    # Calculate the Mean Absolute Error (MAE) for this fold
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Once all folds are done, we'll calculate the average MAE to see our model's overall performance.
average_mae = np.mean(mae_scores)

print(f"Mean Absolute Errors for each fold: {mae_scores}")
print(f"Averaged Mean Absolute Error across all folds: {average_mae:.4f}")

In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt
import seaborn as sns

# Getting the feature importances from the trained Random Forest model.
# This tells us which features the model found most influential in making predictions.
feature_importances = model.feature_importances_
features = X.columns

# Let's create a DataFrame to easily sort and visualize these importances.
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Now, let's plot them to see which features stand out!
plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Importance (higher means more impact)')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Task 2: Write your code here:

import matplotlib.pyplot as plt
import seaborn as sns

# We'll use the predictions from the last fold of our KFold cross-validation.
# This plot helps us understand the distribution of the delivery times our model is predicting.
plt.figure(figsize=(10, 6))
sns.histplot(y_pred, kde=True, bins=30, color='orange')
plt.title('Distribution of Predicted Delivery Times')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Install CatBoost
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor # Import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

# Ensure X and y are defined from previous steps (X_scaled and y)
# X = X_scaled
# y = y

# Set up KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store MAE scores for the ensemble predictions
ensemble_mae_scores = []

# Loop through each fold
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize and train RandomForestRegressor
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train, y_train)
    rf_predictions = rf_model.predict(X_test)

    # Initialize and train CatBoostRegressor
    # verbose=0 to suppress training output for cleanliness
    # random_seed for reproducibility
    cat_model = CatBoostRegressor(random_seed=42, verbose=0, allow_writing_files=False)
    cat_model.fit(X_train, y_train)
    cat_predictions = cat_model.predict(X_test)

    # Average the predictions from both models
    averaged_predictions = (rf_predictions + cat_predictions) / 2

    # Calculate MAE for the averaged predictions
    mae = mean_absolute_error(y_test, averaged_predictions)
    ensemble_mae_scores.append(mae)

# Calculate the average MAE across all folds for the ensemble model
average_ensemble_mae = np.mean(ensemble_mae_scores)

print(f"Mean Absolute Errors for ensemble in each fold: {ensemble_mae_scores}")
print(f"Averaged Mean Absolute Error for Ensemble across all folds: {average_ensemble_mae:.4f}")